__tasks:__
- Converting `bed12` annotations from [HERVd: the Human Endogenous RetroViruses Database](https://herv.img.cas.cz/) to `gtf`
- Build alignment index for STAR and Salmon algorithms.

In [2]:
import shutil
import os
import glob
import gzip
import subprocess

In [3]:
# path_to_files = '/data_gilbert/home/aarab/tools/HERVs/files'
# path_to_files = '/home/abea/genomes/hg38/HERVs/files'
path_to_files = '/hafez/shokatlab/genomes/hg38/HERVs/files'

## Convert `bed12` to `gtf`

https://github.com/CGATOxford/cgat/blob/master/CGAT/scripts/bed2gff.py

In [4]:
# Error in `colnames<-`(`*tmp*`, value = c(paste(samplenames, "input", sep = "-"), : 
# attempt to set 'colnames' on an object with less than two dimensions

def _fix_gtf_headers(gtf_file, overwrite=False, add_header=False, verbose=True):
    """
    Fix GTF file headers by:
    1. Removing comment lines (#)
    2. Adding proper GTF header (if add_header is True)
    3. Saving as .c.gtf (cleaned) [if overwrite is False]
    """
    gtf_out = gtf_file.replace('.gtf', '.c.gtf')
    
    # Read, filter, and write
    with open(gtf_file, 'rt') as infile, open(gtf_out, 'wt') as outfile:
        if add_header:
            # Write header once
            outfile.write('\t'.join(['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']) + '\n')
        
        # Write non-comment lines
        for line in infile:
            if not line.startswith('#'):
                outfile.write(line)

    if overwrite:
        subprocess.run(f'mv {gtf_out} {gtf_file}', shell=True)
        gtf_out = gtf_file  # Update output file name if overwriting

    if verbose:
        print(f"Processed: {gtf_file} -> {gtf_out}")

In [5]:
def convert_bed12_to_gtf(BED, verbose=True, overwrite=False):
    """
    Convert a BED12 file to GTF format.
    """

    GTF = BED.replace('.bed.gz','.gtf')

    if verbose:
        # test if the BED file exists
        if not os.path.exists(BED):
            raise FileNotFoundError("BED file not found")

        # test if the GTF file already exists
        if os.path.exists(GTF):
            if not overwrite:
                raise FileExistsError("GTF file already exists")
            if overwrite:
                print(f"Warning: GTF file {GTF} already exists. Overwriting...")

        # test if cgat is installed
        if not shutil.which("cgat"):
            raise FileNotFoundError("cgat is not installed")

    cm = f'zcat {BED} | cgat bed2gff --as-gtf - > {GTF}'

    if verbose:
        print(f"Running command: {cm} ...", end='')
    
    subprocess.run(cm,shell=True)

    _fix_gtf_headers(gtf_file=GTF, overwrite=True, verbose=verbose)

    if verbose:
        print(f"Done. Output GTF: {GTF}")

---

### erv

In [42]:
convert_bed12_to_gtf(
    BED = f'{path_to_files}/package-entities-erv.bed.gz',
    overwrite = True,
    verbose = True,
)

Running command: zcat /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-erv.bed.gz | cgat bed2gff --as-gtf - > /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-erv.gtf ...Processed: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-erv.gtf -> /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-erv.gtf
Done. Output GTF: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-erv.gtf


In [45]:
%%bash
# note: use gencode.v50 fasta

gtf="files/package-entities-erv.gtf"

genome=${gtf/.gtf/}

echo ${genome}

mkdir ${genome}_star_index

STAR --runThreadN 50 \
    --runMode genomeGenerate \
    --genomeDir ${genome}_star_index \
    --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa \
    --sjdbGTFfile ${gtf} \
    --sjdbOverhang 99

files/package-entities-erv
	/home/abe/miniforge3/envs/alignment/bin/STAR-avx2 --runThreadN 50 --runMode genomeGenerate --genomeDir files/package-entities-erv_star_index --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa --sjdbGTFfile files/package-entities-erv.gtf --sjdbOverhang 99
	STAR version: 2.7.11b   compiled: 2025-11-14T12:06:42+0000 :/opt/conda/conda-bld/star_1763121846936/work/source
Aug 04 16:03:25 ..... started STAR run
Aug 04 16:03:25 ... starting to generate Genome files
Aug 04 16:04:26 ..... processing annotations GTF
Aug 04 16:04:45 ... starting to sort Suffix Array. This may take a long time...
Aug 04 16:05:00 ... sorting Suffix Array chunks and saving them to disk...
Aug 04 16:17:34 ... loading chunks from disk, packing SA...
Aug 04 16:25:18 ... finished generating suffix array
Aug 04 16:25:18 ... generating Suffix Array index
Aug 04 16:27:45 ... completed Suffix Array index
Aug 04 16:27:45 ... writing Genome to disk ...
Aug 

### line

In [46]:
convert_bed12_to_gtf(
    BED = f'{path_to_files}/package-entities-line.bed.gz',
    overwrite = True,
    verbose = True,
)

Running command: zcat /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-line.bed.gz | cgat bed2gff --as-gtf - > /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-line.gtf ...Processed: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-line.gtf -> /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-line.gtf
Done. Output GTF: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-line.gtf


In [47]:
%%bash
# note: use gencode.v50 fasta

gtf="files/package-entities-line.gtf"

genome=${gtf/.gtf/}

echo ${genome}

mkdir ${genome}_star_index

STAR --runThreadN 50 \
    --runMode genomeGenerate \
    --genomeDir ${genome}_star_index \
    --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa \
    --sjdbGTFfile ${gtf} \
    --sjdbOverhang 99

files/package-entities-line
	/home/abe/miniforge3/envs/alignment/bin/STAR-avx2 --runThreadN 50 --runMode genomeGenerate --genomeDir files/package-entities-line_star_index --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa --sjdbGTFfile files/package-entities-line.gtf --sjdbOverhang 99
	STAR version: 2.7.11b   compiled: 2025-11-14T12:06:42+0000 :/opt/conda/conda-bld/star_1763121846936/work/source
Aug 04 16:39:18 ..... started STAR run
Aug 04 16:39:18 ... starting to generate Genome files
Aug 04 16:40:22 ..... processing annotations GTF
Aug 04 16:40:46 ... starting to sort Suffix Array. This may take a long time...
Aug 04 16:41:00 ... sorting Suffix Array chunks and saving them to disk...
Aug 04 16:54:01 ... loading chunks from disk, packing SA...
Aug 04 17:01:47 ... finished generating suffix array
Aug 04 17:01:47 ... generating Suffix Array index
Aug 04 17:04:40 ... completed Suffix Array index
Aug 04 17:04:40 ... writing Genome to disk ...
A

### sine

In [48]:
convert_bed12_to_gtf(
    BED = f'{path_to_files}/package-entities-sine.bed.gz',
    overwrite = True,
    verbose = True
)

Running command: zcat /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-sine.bed.gz | cgat bed2gff --as-gtf - > /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-sine.gtf ...Processed: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-sine.gtf -> /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-sine.gtf
Done. Output GTF: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-sine.gtf


In [49]:
%%bash
# note: use gencode.v50 fasta

gtf="files/package-entities-sine.gtf"

genome=${gtf/.gtf/}

echo ${genome}

mkdir ${genome}_star_index

STAR --runThreadN 50 \
    --runMode genomeGenerate \
    --genomeDir ${genome}_star_index \
    --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa \
    --sjdbGTFfile ${gtf} \
    --sjdbOverhang 99

files/package-entities-sine
	/home/abe/miniforge3/envs/alignment/bin/STAR-avx2 --runThreadN 50 --runMode genomeGenerate --genomeDir files/package-entities-sine_star_index --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa --sjdbGTFfile files/package-entities-sine.gtf --sjdbOverhang 99
	STAR version: 2.7.11b   compiled: 2025-11-14T12:06:42+0000 :/opt/conda/conda-bld/star_1763121846936/work/source
Aug 04 17:24:46 ..... started STAR run
Aug 04 17:24:46 ... starting to generate Genome files
Aug 04 17:25:47 ..... processing annotations GTF
Aug 04 17:26:13 ... starting to sort Suffix Array. This may take a long time...
Aug 04 17:26:27 ... sorting Suffix Array chunks and saving them to disk...
Aug 04 17:40:26 ... loading chunks from disk, packing SA...
Aug 04 17:48:46 ... finished generating suffix array
Aug 04 17:48:46 ... generating Suffix Array index
Aug 04 17:51:31 ... completed Suffix Array index
Aug 04 17:51:31 ... writing Genome to disk ...
A

### retroposon

In [50]:
convert_bed12_to_gtf(
    BED = f'{path_to_files}/package-entities-retroposon.bed.gz',
    overwrite = True,
    verbose = True
)

Running command: zcat /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-retroposon.bed.gz | cgat bed2gff --as-gtf - > /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-retroposon.gtf ...Processed: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-retroposon.gtf -> /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-retroposon.gtf
Done. Output GTF: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-retroposon.gtf


In [51]:
%%bash
# note: use gencode.v50 fasta

gtf="files/package-entities-retroposon.gtf"

genome=${gtf/.gtf/}

echo ${genome}

mkdir ${genome}_star_index

STAR --runThreadN 50 \
    --runMode genomeGenerate \
    --genomeDir ${genome}_star_index \
    --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa \
    --sjdbGTFfile ${gtf} \
    --sjdbOverhang 99

files/package-entities-retroposon
	/home/abe/miniforge3/envs/alignment/bin/STAR-avx2 --runThreadN 50 --runMode genomeGenerate --genomeDir files/package-entities-retroposon_star_index --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa --sjdbGTFfile files/package-entities-retroposon.gtf --sjdbOverhang 99
	STAR version: 2.7.11b   compiled: 2025-11-14T12:06:42+0000 :/opt/conda/conda-bld/star_1763121846936/work/source
Aug 04 18:05:42 ..... started STAR run
Aug 04 18:05:42 ... starting to generate Genome files
Aug 04 18:06:43 ..... processing annotations GTF
Aug 04 18:06:59 ... starting to sort Suffix Array. This may take a long time...
Aug 04 18:07:13 ... sorting Suffix Array chunks and saving them to disk...
Aug 04 18:20:24 ... loading chunks from disk, packing SA...
Aug 04 18:28:13 ... finished generating suffix array
Aug 04 18:28:13 ... generating Suffix Array index
Aug 04 18:31:07 ... completed Suffix Array index
Aug 04 18:31:07 ... writing Ge

### satellite

In [52]:
convert_bed12_to_gtf(
    BED = f'{path_to_files}/package-entities-satellite.bed.gz',
    overwrite = True,
    verbose = True
)

Running command: zcat /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-satellite.bed.gz | cgat bed2gff --as-gtf - > /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-satellite.gtf ...Processed: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-satellite.gtf -> /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-satellite.gtf
Done. Output GTF: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-satellite.gtf


In [53]:
%%bash
# note: use gencode.v50 fasta

gtf="files/package-entities-satellite.gtf"

genome=${gtf/.gtf/}

echo ${genome}

mkdir ${genome}_star_index

STAR --runThreadN 50 \
    --runMode genomeGenerate \
    --genomeDir ${genome}_star_index \
    --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa \
    --sjdbGTFfile ${gtf} \
    --sjdbOverhang 99

files/package-entities-satellite
	/home/abe/miniforge3/envs/alignment/bin/STAR-avx2 --runThreadN 50 --runMode genomeGenerate --genomeDir files/package-entities-satellite_star_index --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa --sjdbGTFfile files/package-entities-satellite.gtf --sjdbOverhang 99
	STAR version: 2.7.11b   compiled: 2025-11-14T12:06:42+0000 :/opt/conda/conda-bld/star_1763121846936/work/source
Aug 04 18:37:03 ..... started STAR run
Aug 04 18:37:03 ... starting to generate Genome files
Aug 04 18:38:05 ..... processing annotations GTF
Aug 04 18:38:21 ... starting to sort Suffix Array. This may take a long time...
Aug 04 18:38:36 ... sorting Suffix Array chunks and saving them to disk...
Aug 04 18:52:34 ... loading chunks from disk, packing SA...
Aug 04 19:00:22 ... finished generating suffix array
Aug 04 19:00:22 ... generating Suffix Array index
Aug 04 19:02:58 ... completed Suffix Array index
Aug 04 19:02:58 ... writing Genom

### rc

Rolling-circle transposable elements (RC transposons), also known as Helitrons, are a distinct class of DNA transposons that replicate via a single-stranded DNA intermediate. Unlike traditional "cut-and-paste" transposons, they use a rolling-circle replication mechanism similar to that of certain bacterial plasmids and viruses.

https://pubmed.ncbi.nlm.nih.gov/11447285/

In [55]:
convert_bed12_to_gtf(
    BED = f'{path_to_files}/package-entities-rc.bed.gz',
    overwrite = True,
    verbose = True
)

Running command: zcat /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-rc.bed.gz | cgat bed2gff --as-gtf - > /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-rc.gtf ...Processed: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-rc.gtf -> /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-rc.gtf
Done. Output GTF: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-rc.gtf


In [1]:
%%bash
# note: use gencode.v50 fasta

gtf="files/package-entities-satellite.gtf"

genome=${gtf/.gtf/}

echo ${genome}

mkdir ${genome}_star_index

STAR --runThreadN 50 \
    --runMode genomeGenerate \
    --genomeDir ${genome}_star_index \
    --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa \
    --sjdbGTFfile ${gtf} \
    --sjdbOverhang 99

files/package-entities-satellite


mkdir: cannot create directory ‘files/package-entities-satellite_star_index’: File exists


	/home/abe/miniforge3/envs/alignment/bin/STAR-avx2 --runThreadN 50 --runMode genomeGenerate --genomeDir files/package-entities-satellite_star_index --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa --sjdbGTFfile files/package-entities-satellite.gtf --sjdbOverhang 99
	STAR version: 2.7.11b   compiled: 2025-11-14T12:06:42+0000 :/opt/conda/conda-bld/star_1763121846936/work/source
Aug 05 00:16:43 ..... started STAR run
Aug 05 00:16:43 ... starting to generate Genome files
Aug 05 00:17:45 ..... processing annotations GTF
Aug 05 00:18:01 ... starting to sort Suffix Array. This may take a long time...
Aug 05 00:18:15 ... sorting Suffix Array chunks and saving them to disk...
Aug 05 00:32:02 ... loading chunks from disk, packing SA...
Aug 05 00:39:52 ... finished generating suffix array
Aug 05 00:39:52 ... generating Suffix Array index
Aug 05 00:42:32 ... completed Suffix Array index
Aug 05 00:42:32 ... writing Genome to disk ...
Aug 05 00:43:00 ...

### snrna

In [57]:
!zcat {path_to_files}/package-entities-snrna.bed.gz | head

chr1	157784	157889	79:snRNA:U6	500	-	157784	157889	67,160,71	1	105	0	 
chr1	516376	516481	236:snRNA:U6	500	-	516376	516481	67,160,71	1	105	0	 
chr1	758233	758338	360:snRNA:U6	500	-	758233	758338	67,160,71	1	105	0	 
chr1	5950255	5950291	3076:snRNA:U2	500	-	5950255	5950291	67,160,71	1	36	0	 
chr1	7796936	7796970	4458:snRNA:U2	500	-	7796936	7796970	67,160,71	1	34	0	 
chr1	7994455	7994519	4641:snRNA:U6	500	-	7994455	7994519	67,160,71	1	64	0	 
chr1	8206434	8206595	4875:snRNA:U1	500	-	8206434	8206595	67,160,71	1	161	0	 
chr1	8292169	8292269	4993:snRNA:U6	500	-	8292169	8292269	67,160,71	1	100	0	 
chr1	8883431	8883522	5391:snRNA:U6	500	-	8883431	8883522	67,160,71	1	91	0	 
chr1	9535389	9535423	6038:snRNA:U2	500	-	9535389	9535423	67,160,71	1	34	0	 

gzip: stdout: Broken pipe


In [6]:
convert_bed12_to_gtf(
    BED = f'{path_to_files}/package-entities-snrna.bed.gz',
    overwrite = True,
    verbose = True
)

Running command: zcat /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-snrna.bed.gz | cgat bed2gff --as-gtf - > /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-snrna.gtf ...Processed: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-snrna.gtf -> /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-snrna.gtf
Done. Output GTF: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-snrna.gtf


In [7]:
%%bash
# note: use gencode.v50 fasta

gtf="files/package-entities-snrna.gtf"

genome=${gtf/.gtf/}

echo ${genome}

mkdir ${genome}_star_index

STAR --runThreadN 50 \
    --runMode genomeGenerate \
    --genomeDir ${genome}_star_index \
    --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa \
    --sjdbGTFfile ${gtf} \
    --sjdbOverhang 99

files/package-entities-snrna
	/home/abe/miniforge3/envs/alignment/bin/STAR-avx2 --runThreadN 50 --runMode genomeGenerate --genomeDir files/package-entities-snrna_star_index --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa --sjdbGTFfile files/package-entities-snrna.gtf --sjdbOverhang 99
	STAR version: 2.7.11b   compiled: 2025-11-14T12:06:42+0000 :/opt/conda/conda-bld/star_1763121846936/work/source
Aug 05 00:46:51 ..... started STAR run
Aug 05 00:46:51 ... starting to generate Genome files
Aug 05 00:47:52 ..... processing annotations GTF
Aug 05 00:48:07 ... starting to sort Suffix Array. This may take a long time...
Aug 05 00:48:21 ... sorting Suffix Array chunks and saving them to disk...
Aug 05 01:02:33 ... loading chunks from disk, packing SA...
Aug 05 01:10:25 ... finished generating suffix array
Aug 05 01:10:25 ... generating Suffix Array index
Aug 05 01:13:00 ... completed Suffix Array index
Aug 05 01:13:00 ... writing Genome to disk ..

### scrna

In [8]:
convert_bed12_to_gtf(
    BED = f'{path_to_files}/package-entities-scrna.bed.gz',
    overwrite = True,
    verbose = True
)

Running command: zcat /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-scrna.bed.gz | cgat bed2gff --as-gtf - > /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-scrna.gtf ...Processed: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-scrna.gtf -> /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-scrna.gtf
Done. Output GTF: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-scrna.gtf


In [9]:
%%bash
# note: use gencode.v50 fasta

gtf="files/package-entities-scrna.gtf"

genome=${gtf/.gtf/}

echo ${genome}

mkdir ${genome}_star_index

STAR --runThreadN 50 \
    --runMode genomeGenerate \
    --genomeDir ${genome}_star_index \
    --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa \
    --sjdbGTFfile ${gtf} \
    --sjdbOverhang 99

files/package-entities-scrna
	/home/abe/miniforge3/envs/alignment/bin/STAR-avx2 --runThreadN 50 --runMode genomeGenerate --genomeDir files/package-entities-scrna_star_index --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa --sjdbGTFfile files/package-entities-scrna.gtf --sjdbOverhang 99
	STAR version: 2.7.11b   compiled: 2025-11-14T12:06:42+0000 :/opt/conda/conda-bld/star_1763121846936/work/source
Aug 05 01:17:20 ..... started STAR run
Aug 05 01:17:20 ... starting to generate Genome files
Aug 05 01:18:20 ..... processing annotations GTF
Aug 05 01:18:35 ... starting to sort Suffix Array. This may take a long time...
Aug 05 01:18:49 ... sorting Suffix Array chunks and saving them to disk...
Aug 05 01:32:05 ... loading chunks from disk, packing SA...
Aug 05 01:39:56 ... finished generating suffix array
Aug 05 01:39:56 ... generating Suffix Array index
Aug 05 01:42:52 ... completed Suffix Array index
Aug 05 01:42:53 ... writing Genome to disk ..

### trna

In [10]:
convert_bed12_to_gtf(
    BED = f'{path_to_files}/package-entities-trna.bed.gz',
    overwrite = True,
    verbose = True
)

Running command: zcat /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-trna.bed.gz | cgat bed2gff --as-gtf - > /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-trna.gtf ...Processed: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-trna.gtf -> /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-trna.gtf
Done. Output GTF: /hafez/shokatlab/genomes/hg38/HERVs/files/package-entities-trna.gtf


In [11]:
%%bash
# note: use gencode.v50 fasta

gtf="files/package-entities-trna.gtf"

genome=${gtf/.gtf/}

echo ${genome}

mkdir ${genome}_star_index

STAR --runThreadN 50 \
    --runMode genomeGenerate \
    --genomeDir ${genome}_star_index \
    --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa \
    --sjdbGTFfile ${gtf} \
    --sjdbOverhang 99

files/package-entities-trna
	/home/abe/miniforge3/envs/alignment/bin/STAR-avx2 --runThreadN 50 --runMode genomeGenerate --genomeDir files/package-entities-trna_star_index --genomeFastaFiles /hafez/shokatlab/genomes/hg38/gencode.v50/GRCh38.primary_assembly.genome.fa --sjdbGTFfile files/package-entities-trna.gtf --sjdbOverhang 99
	STAR version: 2.7.11b   compiled: 2025-11-14T12:06:42+0000 :/opt/conda/conda-bld/star_1763121846936/work/source
Aug 05 01:47:10 ..... started STAR run
Aug 05 01:47:10 ... starting to generate Genome files
Aug 05 01:48:12 ..... processing annotations GTF
Aug 05 01:48:28 ... starting to sort Suffix Array. This may take a long time...
Aug 05 01:48:42 ... sorting Suffix Array chunks and saving them to disk...
Aug 05 02:02:58 ... loading chunks from disk, packing SA...
Aug 05 02:10:46 ... finished generating suffix array
Aug 05 02:10:46 ... generating Suffix Array index
Aug 05 02:13:41 ... completed Suffix Array index
Aug 05 02:13:41 ... writing Genome to disk ...
A

### ...

## bed2fa

In [ ]:
# !zcat files/package-entities-erv.gff3.gz | head -n 40

In [ ]:
# !zcat files/package-entities-erv.bed.gz | head -n 40

In [ ]:
# !zcat files/package-entities-erv.gtf.gz | head -n 40

In [ ]:
def convert_bed12_to_fasta(BED,ref_fa):
    FASTA = BED.replace('.bed.gz','.bed.fa.gz')
    cm = f'zcat {BED} | bedtools getfasta -split -name+ -fi {ref_fa} -bed - | gzip > {FASTA}'
    subprocess.run(cm,shell=True)

In [ ]:
# !wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_46/GRCh38.primary_assembly.genome.fa.gz
# mv GRCh38.primary_assembly.genome.fa.gz ../hg38/gencode.v46/
# !gunzip {ref_fa}.gz

In [ ]:
ref_fa='/hafez/shokatlab/genomes/hg38/gencode.v46/GRCh38.primary_assembly.genome.fa'

# check if ref_fa exists
if not os.path.exists(ref_fa):
    raise FileNotFoundError(f"Reference genome file not found: {ref_fa}")

for loop ...

In [ ]:
for name in ['erv','line','rc','retroposon','satellite','scrna','sine','snrna','trna']:

    print(f"Converting BED12 to FASTA for: {name} ...")
    
    convert_bed12_to_fasta(
        BED = f'{path_to_files}/package-entities-{name}.bed.gz',
        ref_fa=ref_fa
    )

    print(f"Done. Output FASTA: {path_to_files}/package-entities-{name}.bed.fa.gz")

## Salmon index

In [ ]:
# def build_salmon_index(FASTA, threads):
#     index=FASTA.replace('.fa.gz','_salmon_index')
#     log=FASTA.replace('.fa.gz','_salmon_index.log')
#     cm = f'salmon index -t {FASTA} -i {index} --threads {threads} > {log} 2>&1'
    
#     print(cm)
    
#     # subprocess.run(cm,shell=True)
# for name in ['erv','line','rc','retroposon','satellite','scrna','sine','snrna','trna']:
#     build_salmon_index(
#         FASTA = f'{path_to_files}/package-entities-{name}.bed.fa.gz',
#         threads = 36
#     )
# cat /home/abea/genomes/hg38/HERVs/files/package-entities-erv.bed_salmon_index.log

## kallisto index

In [ ]:

# # def build_kallisto_index(FASTA):
# #     index=FASTA.replace('.fa.gz','.idx')
# #     cm = f'kallisto index -i {index} {FASTA}'
# #     print(cm)
# #     # subprocess.run(cm,shell=True)
# for name in ['erv','line','rc','retroposon','satellite','scrna','sine','snrna','trna']:
#     build_kallisto_index(
#         FASTA = f'{path_to_files}/package-entities-{name}.bed.fa.gz',
#         threads = 36
#     )


## 

In [12]:
%load_ext watermark
%watermark
%watermark -iversion

Last updated: 2026-08-05T02:18:01.443660-07:00

Python implementation: CPython
Python version       : 3.12.13
IPython version      : 9.15.0

Compiler    : GCC 14.3.0
OS          : Linux
Release     : 4.18.0-513.9.1.el8_9.x86_64
Machine     : x86_64
Processor   : 
CPU cores   : 64
Architecture: 64bit

Email: rsion

Python implementation: CPython
Python version       : 3.12.13
IPython version      : 9.15.0



In [13]:
!date

Wed Aug  5 02:18:01 PDT 2026


___